# 🎵 Spotify Search API - Hướng dẫn truy xuất dữ liệu theo Batch

Notebook này hướng dẫn chi tiết cách sử dụng **Search endpoint** (`GET /search`) của Spotify Web API để truy xuất dữ liệu theo từng batch (lô) một cách hiệu quả.

## 📋 Mục lục
1. [Cài đặt & Thiết lập](#1-cài-đặt--thiết-lập)
2. [Xác thực (Authentication)](#2-xác-thực-authentication)
3. [Tìm hiểu Search Endpoint](#3-tìm-hiểu-search-endpoint)
4. [Truy xuất dữ liệu cơ bản](#4-truy-xuất-dữ-liệu-cơ-bản)
5. [Truy xuất theo Batch (Pagination)](#5-truy-xuất-theo-batch-pagination)
6. [Batch nâng cao: Nhiều loại dữ liệu](#6-batch-nâng-cao-nhiều-loại-dữ-liệu)
7. [Xử lý Rate Limiting & Retry](#7-xử-lý-rate-limiting--retry)
8. [Xuất dữ liệu ra CSV/JSON](#8-xuất-dữ-liệu-ra-csvjson)
9. [Ví dụ thực tế: Thu thập dữ liệu lớn](#9-ví-dụ-thực-tế-thu-thập-dữ-liệu-lớn)

## 📌 Yêu cầu
- File `.env` chứa `SPOTIFY_CLIENT_ID` và `SPOTIFY_CLIENT_SECRET`
- Cài đặt thư viện: `requests`, `python-dotenv`, `pandas`

## 🔗 Tài liệu tham khảo
- [Spotify Search API Reference](https://developer.spotify.com/documentation/web-api/reference/search)

---
## 1. Cài đặt & Thiết lập

In [11]:
# Cài đặt thư viện (chỉ cần chạy 1 lần)
!pip install requests python-dotenv pandas


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import os
import sys
import json
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime


# Load biến môi trường
load_dotenv()

SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
BASE_URL = "https://api.spotify.com/v1"

print(f"✅ Client ID: {SPOTIFY_CLIENT_ID[:8]}..." if SPOTIFY_CLIENT_ID else "❌ Thiếu SPOTIFY_CLIENT_ID")
print(f"✅ Client Secret: {'*' * 8}..." if SPOTIFY_CLIENT_SECRET else "❌ Thiếu SPOTIFY_CLIENT_SECRET")

✅ Client ID: 53bc91a1...
✅ Client Secret: ********...


---
## 2. Xác thực (Authentication)

Sử dụng **Client Credentials Flow** để lấy access token.

| Thông tin | Chi tiết |
|-----------|----------|
| **URL** | `POST https://accounts.spotify.com/api/token` |
| **grant_type** | `client_credentials` |
| **Token hết hạn** | Sau 3600 giây (1 giờ) |

In [ ]:
def get_access_token():
    """
    Lấy Access Token bằng Client Credentials Flow.
    Token này chỉ cho phép truy cập dữ liệu công khai.
    
    Returns:
        str: Access token
    """
    auth_url = "https://accounts.spotify.com/api/token"
    
    response = requests.post(auth_url, data={
        "grant_type": "client_credentials",
        "client_id": 'client_id',
        "client_secret": 'client_secret',
    })
    
    if response.status_code == 200:
        token_data = response.json()
        print("✅ Lấy token thành công!")
        print(f"   Token type: {token_data['token_type']}")
        print(f"   Hết hạn sau: {token_data['expires_in']} giây ({token_data['expires_in']//60} phút)")
        return token_data["access_token"]
    else:
        print(f"❌ Lỗi: {response.status_code}")
        print(response.json())
        return None

ACCESS_TOKEN = get_access_token()
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}

✅ Lấy token thành công!
   Token type: Bearer
   Hết hạn sau: 3600 giây (60 phút)


---
## 3. Tìm hiểu Search Endpoint

### 3.1 Thông tin endpoint

| Thông tin | Chi tiết |
|-----------|----------|
| **Method** | `GET` |
| **URL** | `https://api.spotify.com/v1/search` |
| **Auth** | Bearer token (không cần scope đặc biệt) |

### 3.2 Tham số (Parameters)

| Param | Loại | Bắt buộc | Mô tả | Giá trị |
|-------|------|----------|-------|--------|
| `q` | query | ✅ Có | Từ khóa tìm kiếm | Bất kỳ chuỗi nào |
| `type` | query | ✅ Có | Loại kết quả muốn tìm | `album`, `artist`, `playlist`, `track`, `show`, `episode`, `audiobook` |
| `market` | query | ❌ Không | Mã quốc gia ISO 3166-1 alpha-2 | VD: `VN`, `US`, `JP` |
| `limit` | query | ❌ Không | Số kết quả mỗi trang | Mặc định: `20`, Min: `0`, Max: `50` |
| `offset` | query | ❌ Không | Vị trí bắt đầu (index) | Mặc định: `0`, Max: `1000` |
| `include_external` | query | ❌ Không | Bao gồm nội dung hosted bên ngoài | `audio` |

### 3.3 Cú pháp tìm kiếm nâng cao (Query Syntax)

| Filter | Ví dụ | Mô tả |
|--------|-------|-------|
| `track:` | `track:Imagine` | Tìm theo tên bài hát |
| `artist:` | `artist:Beatles` | Tìm theo tên nghệ sĩ |
| `album:` | `album:Abbey Road` | Tìm theo tên album |
| `year:` | `year:2024` | Lọc theo năm |
| `year:` | `year:2020-2024` | Lọc theo khoảng năm |
| `genre:` | `genre:rock` | Lọc theo thể loại |
| `isrc:` | `isrc:USUM71703861` | Tìm theo mã ISRC |
| `tag:new` | `tag:new` | Album mới (2 tuần gần nhất) |
| `tag:hipster` | `tag:hipster` | Album ít phổ biến nhất |
| Kết hợp | `track:Imagine artist:Lennon` | Kết hợp nhiều filter |

### 3.4 Giới hạn quan trọng

> ⚠️ **`offset` tối đa là 1000**. Nghĩa là bạn chỉ có thể truy cập tối đa **1000 + limit** kết quả cho 1 truy vấn tìm kiếm. Đây là giới hạn của Spotify API, không phải lỗi code.

---
## 4. Truy xuất dữ liệu cơ bản

Trước khi truy xuất theo batch, hãy hiểu cấu trúc response trước.

In [18]:
# === Request tìm kiếm đơn giản ===
response = requests.get(
    f"{BASE_URL}/search",
    headers=HEADERS,
    params={
        "q": "V-Pop",       # Từ khóa tìm kiếm
        "type": "track",    # Loại: chỉ tìm tracks
        "limit": 5,         # Lấy 5 kết quả
        "offset": 0,        # Bắt đầu từ vị trí 0
        "market": "VN"      # Thị trường Việt Nam
    }
)

data = response.json()
print(json.dumps(data,indent=2))

{
  "tracks": {
    "href": "https://api.spotify.com/v1/search?offset=0&limit=5&query=V-Pop&type=track&market=VN",
    "limit": 5,
    "next": "https://api.spotify.com/v1/search?offset=5&limit=5&query=V-Pop&type=track&market=VN",
    "offset": 0,
    "previous": null,
    "total": 7,
    "items": [
      {
        "album": {
          "album_type": "single",
          "artists": [
            {
              "external_urls": {
                "spotify": "https://open.spotify.com/artist/0IdAjS2LRieBR3gzoazdAw"
              },
              "href": "https://api.spotify.com/v1/artists/0IdAjS2LRieBR3gzoazdAw",
              "id": "0IdAjS2LRieBR3gzoazdAw",
              "name": "MIN",
              "type": "artist",
              "uri": "spotify:artist:0IdAjS2LRieBR3gzoazdAw"
            }
          ],
          "external_urls": {
            "spotify": "https://open.spotify.com/album/5iu3qeOQixQ02tRduRp4VV"
          },
          "href": "https://api.spotify.com/v1/albums/5iu3qeOQixQ02tRd

In [ ]:
# === Phân tích Paging Object ===
# Response search luôn trả về Paging Object cho mỗi type

tracks_paging = data['tracks']

print("📄 PAGING OBJECT - Các trường quan trọng cho việc batch:")
print(f"   ├── href:     {tracks_paging['href'][:80]}...")
print(f"   ├── total:    {tracks_paging['total']}  ← Tổng số kết quả tìm được")
print(f"   ├── limit:    {tracks_paging['limit']}  ← Số kết quả mỗi trang")
print(f"   ├── offset:   {tracks_paging['offset']}  ← Vị trí bắt đầu trang hiện tại")
print(f"   ├── next:     {tracks_paging['next'][:80] if tracks_paging['next'] else 'None'}  ← URL trang tiếp")
print(f"   ├── previous: {tracks_paging['previous']}  ← URL trang trước")
print(f"   └── items:    [{len(tracks_paging['items'])} items]  ← Dữ liệu trang hiện tại")

print(f"\n🎵 Kết quả tìm được (trang 1):")
for i, track in enumerate(tracks_paging['items']):
    artists = ', '.join([a['name'] for a in track['artists']])
    print(f"   {i+1}. {track['name']} - {artists} (Popularity: {track['popularity']})")

### 📖 Giải thích Paging Object

Mọi endpoint trả về danh sách đều sử dụng **Paging Object** để phân trang:

| Trường | Kiểu | Mô tả | Vai trò trong Batch |
|--------|------|-------|--------------------|
| `href` | string | URL API trang hiện tại | Debug |
| `items` | array | Dữ liệu trang hiện tại | **Dữ liệu cần thu thập** |
| `limit` | integer | Số items tối đa / trang | **Kích thước batch** |
| `next` | string/null | URL trang tiếp theo | **Điều kiện dừng** (null = hết) |
| `offset` | integer | Vị trí bắt đầu | **Theo dõi tiến trình** |
| `previous` | string/null | URL trang trước | Ít dùng |
| `total` | integer | Tổng số items | **Tính số batch cần thiết** |

---
## 5. Truy xuất theo Batch (Pagination)

### Nguyên lý hoạt động

```
Batch 1: offset=0,  limit=50  → items[0..49]
Batch 2: offset=50, limit=50  → items[50..99]
Batch 3: offset=100,limit=50  → items[100..149]
...tiếp tục cho đến khi next=null hoặc offset > 1000
```

### 5.1 Cách 1: Sử dụng offset tăng dần (Thủ công)

In [ ]:
def search_batch_manual(query, search_type="track", market="VN", batch_size=50, max_items=200):
    """
    Truy xuất dữ liệu theo batch bằng cách tăng offset thủ công.
    
    Args:
        query (str): Từ khóa tìm kiếm
        search_type (str): Loại kết quả (track, artist, album, playlist, ...)
        market (str): Mã quốc gia ISO 3166-1 alpha-2
        batch_size (int): Số items mỗi batch (max 50)
        max_items (int): Số items tối đa muốn lấy
    
    Returns:
        list: Danh sách tất cả items thu thập được
    """
    all_items = []
    offset = 0
    batch_num = 0
    
    # Tên key trong response tương ứng với type (track → tracks, artist → artists, ...)
    result_key = search_type + "s"
    
    print(f"🔍 Bắt đầu tìm kiếm: q='{query}', type='{search_type}'")
    print(f"   Batch size: {batch_size}, Max items: {max_items}")
    print("=" * 60)
    
    while offset < max_items:
        batch_num += 1
        
        # Tính limit cho batch hiện tại (batch cuối có thể nhỏ hơn)
        current_limit = min(batch_size, max_items - offset)
        
        # Gọi API
        response = requests.get(
            f"{BASE_URL}/search",
            headers=HEADERS,
            params={
                "q": query,
                "type": search_type,
                "limit": current_limit,
                "offset": offset,
                "market": market
            }
        )
        
        if response.status_code != 200:
            print(f"   ❌ Batch {batch_num}: Lỗi {response.status_code}")
            break
        
        data = response.json()
        paging = data[result_key]
        items = paging['items']
        total = paging['total']
        
        all_items.extend(items)
        
        print(f"   📦 Batch {batch_num}: offset={offset}, lấy được {len(items)} items "
              f"(tổng cộng: {len(all_items)}/{min(total, max_items)})")
        
        # Điều kiện dừng:
        # 1. Không còn trang tiếp theo
        # 2. Đã lấy đủ số lượng mong muốn
        # 3. offset vượt quá tổng kết quả
        # 4. offset vượt quá giới hạn API (1000)
        if paging['next'] is None:
            print(f"   ✅ Hết dữ liệu (next=null)")
            break
        
        if len(all_items) >= max_items:
            print(f"   ✅ Đã đạt max_items={max_items}")
            break
            
        if offset + current_limit >= 1000:
            print(f"   ⚠️ Đạt giới hạn offset API (1000)")
            break
        
        # Tăng offset cho batch tiếp theo
        offset += current_limit
        
        # Delay nhỏ để tránh rate limiting
        time.sleep(0.2)
    
    print("=" * 60)
    print(f"✅ Hoàn tất! Tổng cộng thu thập được: {len(all_items)} items")
    return all_items

In [ ]:
# === Ví dụ: Lấy 200 bài hát V-Pop ===
tracks = search_batch_manual(
    query="V-Pop",
    search_type="track",
    market="VN",
    batch_size=50,   # Mỗi batch lấy 50 items (max)
    max_items=200    # Tổng cộng muốn lấy 200 items
)

# Hiển thị 10 kết quả đầu
print(f"\n🎵 10 bài hát đầu tiên:")
for i, t in enumerate(tracks[:10]):
    artists = ', '.join([a['name'] for a in t['artists']])
    print(f"   {i+1}. {t['name']} - {artists} (Popularity: {t['popularity']})")

### 5.2 Cách 2: Sử dụng trường `next` (Khuyến nghị)

Spotify trả về trường `next` chứa URL đầy đủ cho trang tiếp theo. Đây là cách **đáng tin cậy nhất** vì:
- Không cần tự tính offset
- API tự xử lý các edge case
- Đúng theo best practice của Spotify

In [ ]:
def search_batch_using_next(query, search_type="track", market="VN", batch_size=50, max_items=200):
    """
    Truy xuất dữ liệu theo batch bằng cách dùng trường 'next' URL.
    
    Cách này đáng tin cậy hơn vì sử dụng URL do API cung cấp,
    thay vì tự tính offset.
    
    Args:
        query (str): Từ khóa tìm kiếm
        search_type (str): Loại kết quả
        market (str): Mã quốc gia
        batch_size (int): Số items mỗi batch (max 50)
        max_items (int): Số items tối đa muốn lấy
    
    Returns:
        tuple: (all_items, total_available)
    """
    all_items = []
    batch_num = 0
    result_key = search_type + "s"
    
    # Request đầu tiên
    url = f"{BASE_URL}/search"
    params = {
        "q": query,
        "type": search_type,
        "limit": min(batch_size, 50),
        "offset": 0,
        "market": market
    }
    
    print(f"🔍 Tìm kiếm: q='{query}', type='{search_type}', market='{market}'")
    print("=" * 60)
    
    while url and len(all_items) < max_items:
        batch_num += 1
        
        # Gọi API - batch đầu dùng params, các batch sau dùng next URL
        if batch_num == 1:
            response = requests.get(url, headers=HEADERS, params=params)
        else:
            response = requests.get(url, headers=HEADERS)
        
        if response.status_code != 200:
            print(f"   ❌ Batch {batch_num}: Lỗi {response.status_code}")
            break
        
        data = response.json()
        paging = data[result_key]
        items = paging['items']
        total = paging['total']
        
        all_items.extend(items)
        
        print(f"   📦 Batch {batch_num}: lấy {len(items)} items "
              f"(tổng: {len(all_items)}/{min(total, max_items)})")
        
        # Lấy URL trang tiếp theo từ response
        url = paging.get('next')  # None nếu hết trang
        
        # Delay nhỏ để tránh rate limiting
        time.sleep(0.2)
    
    print("=" * 60)
    print(f"✅ Hoàn tất! Thu thập: {len(all_items)} items (API có tổng: {total})")
    return all_items, total

In [ ]:
# === Ví dụ: Lấy 150 bài hát K-Pop ===
kpop_tracks, total_available = search_batch_using_next(
    query="genre:k-pop",
    search_type="track",
    market="VN",
    batch_size=50,
    max_items=150
)

print(f"\n📊 Thống kê:")
print(f"   Tổng kết quả trên Spotify: {total_available}")
print(f"   Đã thu thập: {len(kpop_tracks)}")
print(f"   Số batch đã gọi: {(len(kpop_tracks) + 49) // 50}")

### 📖 So sánh 2 cách truy xuất batch

| Tiêu chí | Cách 1 (Offset thủ công) | Cách 2 (Dùng `next`) |
|----------|-------------------------|----------------------|
| **Độ tin cậy** | Trung bình | Cao |
| **Linh hoạt** | Có thể nhảy offset | Chỉ đi tuần tự |
| **Phức tạp** | Phải tự tính offset | Đơn giản hơn |
| **Edge cases** | Tự xử lý | API xử lý |
| **Khuyến nghị** | Khi cần nhảy trang | ✅ Mặc định nên dùng |

---
## 6. Batch nâng cao: Nhiều loại dữ liệu

Bạn có thể tìm kiếm nhiều loại dữ liệu cùng lúc bằng cách truyền nhiều `type`.

In [ ]:
def search_batch_multi_type(query, search_types, market="VN", batch_size=50, max_items_per_type=100):
    """
    Truy xuất batch cho nhiều loại dữ liệu cùng lúc.
    
    LƯU Ý: Khi tìm nhiều type cùng lúc, mỗi type sẽ có paging riêng,
    nhưng chúng chia sẻ cùng offset/limit trong cùng 1 request.
    
    Chiến lược tốt hơn: Tìm riêng từng type để kiểm soát tốt hơn.
    
    Args:
        query (str): Từ khóa tìm kiếm
        search_types (list): Danh sách loại, VD: ['track', 'artist', 'album']
        market (str): Mã quốc gia
        batch_size (int): Số items mỗi batch
        max_items_per_type (int): Số items tối đa cho mỗi loại
    
    Returns:
        dict: {type_name: [items]}
    """
    results = {}
    
    for search_type in search_types:
        print(f"\n{'='*60}")
        print(f"🔍 Đang tìm kiếm loại: {search_type.upper()}")
        print(f"{'='*60}")
        
        items, total = search_batch_using_next(
            query=query,
            search_type=search_type,
            market=market,
            batch_size=batch_size,
            max_items=max_items_per_type
        )
        
        results[search_type] = {
            'items': items,
            'total_on_spotify': total,
            'collected': len(items)
        }
    
    return results

In [ ]:
# === Ví dụ: Tìm "Sơn Tùng" - lấy tracks, artists, albums ===
multi_results = search_batch_multi_type(
    query="Sơn Tùng M-TP",
    search_types=["track", "artist", "album"],
    market="VN",
    batch_size=50,
    max_items_per_type=100
)

# Tổng kết
print(f"\n{'='*60}")
print("📊 TỔNG KẾT:")
for type_name, info in multi_results.items():
    print(f"   {type_name}: thu thập {info['collected']}/{info['total_on_spotify']} items")

---
## 7. Xử lý Rate Limiting & Retry

Spotify API có giới hạn số request. Khi vượt quá, API trả về **HTTP 429 (Too Many Requests)** kèm header `Retry-After` cho biết cần chờ bao lâu.

### Chiến lược xử lý:
1. **Delay giữa các request**: `time.sleep(0.2)` (200ms)
2. **Retry khi gặp 429**: Chờ theo `Retry-After` header
3. **Exponential backoff**: Tăng thời gian chờ sau mỗi lần retry

In [ ]:
def search_batch_robust(query, search_type="track", market="VN", 
                         batch_size=50, max_items=500, 
                         delay=0.3, max_retries=3):
    """
    Truy xuất batch với xử lý lỗi đầy đủ:
    - Rate limiting (429)
    - Retry với exponential backoff
    - Token hết hạn
    - Lỗi mạng
    
    Args:
        query (str): Từ khóa tìm kiếm
        search_type (str): Loại kết quả
        market (str): Mã quốc gia
        batch_size (int): Số items mỗi batch (max 50)
        max_items (int): Số items tối đa
        delay (float): Thời gian chờ giữa các batch (giây)
        max_retries (int): Số lần retry tối đa cho mỗi batch
    
    Returns:
        tuple: (all_items, metadata)
    """
    all_items = []
    batch_num = 0
    result_key = search_type + "s"
    total_available = 0
    
    # Metadata để theo dõi quá trình
    metadata = {
        'query': query,
        'type': search_type,
        'market': market,
        'start_time': datetime.now().isoformat(),
        'batches': 0,
        'retries': 0,
        'errors': []
    }
    
    # URL ban đầu
    url = f"{BASE_URL}/search"
    params = {
        "q": query,
        "type": search_type,
        "limit": min(batch_size, 50),
        "offset": 0,
        "market": market
    }
    is_first = True
    
    print(f"🚀 Bắt đầu truy xuất batch")
    print(f"   Query: '{query}' | Type: {search_type} | Market: {market}")
    print(f"   Batch size: {batch_size} | Max items: {max_items} | Delay: {delay}s")
    print("=" * 70)
    
    while url and len(all_items) < max_items:
        batch_num += 1
        retry_count = 0
        success = False
        
        while retry_count <= max_retries and not success:
            try:
                # Gọi API
                if is_first:
                    response = requests.get(url, headers=HEADERS, params=params, timeout=10)
                    is_first = False
                else:
                    response = requests.get(url, headers=HEADERS, timeout=10)
                
                # Xử lý Rate Limiting
                if response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 5))
                    print(f"   ⏳ Rate limited! Chờ {retry_after}s...")
                    time.sleep(retry_after)
                    retry_count += 1
                    metadata['retries'] += 1
                    continue
                
                # Xử lý Token hết hạn
                if response.status_code == 401:
                    print(f"   🔄 Token hết hạn, đang refresh...")
                    global ACCESS_TOKEN, HEADERS
                    ACCESS_TOKEN = get_access_token()
                    HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
                    retry_count += 1
                    continue
                
                # Xử lý lỗi khác
                if response.status_code != 200:
                    error_msg = f"HTTP {response.status_code}"
                    print(f"   ❌ Batch {batch_num}: {error_msg}")
                    metadata['errors'].append({'batch': batch_num, 'error': error_msg})
                    retry_count += 1
                    time.sleep(2 ** retry_count)  # Exponential backoff
                    continue
                
                # Thành công!
                data = response.json()
                paging = data[result_key]
                items = paging['items']
                total_available = paging['total']
                
                all_items.extend(items)
                metadata['batches'] += 1
                
                # Progress bar đơn giản
                target = min(total_available, max_items)
                progress = len(all_items) / target * 100 if target > 0 else 100
                bar = '█' * int(progress // 5) + '░' * (20 - int(progress // 5))
                print(f"   📦 Batch {batch_num}: [{bar}] {progress:.0f}% "
                      f"({len(all_items)}/{target})")
                
                # Lấy URL tiếp theo
                url = paging.get('next')
                success = True
                
            except requests.exceptions.Timeout:
                print(f"   ⏰ Timeout! Retry ({retry_count + 1}/{max_retries})...")
                retry_count += 1
                time.sleep(2 ** retry_count)
                
            except requests.exceptions.ConnectionError:
                print(f"   🌐 Lỗi kết nối! Retry ({retry_count + 1}/{max_retries})...")
                retry_count += 1
                time.sleep(2 ** retry_count)
        
        if not success:
            print(f"   ❌ Batch {batch_num}: Hết retry, bỏ qua.")
            break
        
        # Delay giữa các batch
        time.sleep(delay)
    
    metadata['end_time'] = datetime.now().isoformat()
    metadata['total_collected'] = len(all_items)
    metadata['total_on_spotify'] = total_available
    
    print("=" * 70)
    print(f"✅ Hoàn tất!")
    print(f"   Đã thu thập: {len(all_items)} items")
    print(f"   Tổng trên Spotify: {total_available}")
    print(f"   Số batch: {metadata['batches']} | Retries: {metadata['retries']}")
    print(f"   Lỗi: {len(metadata['errors'])}")
    
    return all_items, metadata

In [ ]:
# === Ví dụ: Lấy 300 bài hát Pop với xử lý lỗi đầy đủ ===
pop_tracks, meta = search_batch_robust(
    query="genre:pop year:2024",
    search_type="track",
    market="VN",
    batch_size=50,
    max_items=300,
    delay=0.3
)

print(f"\n📋 Metadata:")
print(json.dumps(meta, indent=2, ensure_ascii=False, default=str))

---
## 8. Xuất dữ liệu ra CSV/JSON

Sau khi thu thập, bạn thường cần lưu dữ liệu để phân tích sau.

In [ ]:
def tracks_to_dataframe(tracks):
    """
    Chuyển danh sách tracks thành DataFrame.
    
    Trích xuất các trường quan trọng từ Track Object.
    
    Args:
        tracks (list): Danh sách Track Objects từ API
    
    Returns:
        pd.DataFrame: DataFrame chứa dữ liệu đã xử lý
    """
    rows = []
    for t in tracks:
        if t is None:
            continue
        row = {
            'track_id': t.get('id', ''),
            'track_name': t.get('name', ''),
            'artists': ', '.join([a['name'] for a in t.get('artists', [])]),
            'artist_ids': ', '.join([a['id'] for a in t.get('artists', [])]),
            'album_name': t.get('album', {}).get('name', ''),
            'album_id': t.get('album', {}).get('id', ''),
            'album_type': t.get('album', {}).get('album_type', ''),
            'release_date': t.get('album', {}).get('release_date', ''),
            'duration_ms': t.get('duration_ms', 0),
            'duration_s': round(t.get('duration_ms', 0) / 1000, 1),
            'popularity': t.get('popularity', 0),
            'explicit': t.get('explicit', False),
            'track_number': t.get('track_number', 0),
            'disc_number': t.get('disc_number', 0),
            'isrc': t.get('external_ids', {}).get('isrc', ''),
            'spotify_url': t.get('external_urls', {}).get('spotify', ''),
            'preview_url': t.get('preview_url', ''),
            'uri': t.get('uri', '')
        }
        rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
def artists_to_dataframe(artists):
    """
    Chuyển danh sách artists thành DataFrame.
    
    Args:
        artists (list): Danh sách Artist Objects từ API
    
    Returns:
        pd.DataFrame
    """
    rows = []
    for a in artists:
        if a is None:
            continue
        row = {
            'artist_id': a.get('id', ''),
            'artist_name': a.get('name', ''),
            'genres': ', '.join(a.get('genres', [])),
            'followers': a.get('followers', {}).get('total', 0),
            'popularity': a.get('popularity', 0),
            'spotify_url': a.get('external_urls', {}).get('spotify', ''),
            'image_url': a.get('images', [{}])[0].get('url', '') if a.get('images') else '',
            'uri': a.get('uri', '')
        }
        rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
def albums_to_dataframe(albums):
    """
    Chuyển danh sách albums thành DataFrame.
    
    Args:
        albums (list): Danh sách Album Objects từ API
    
    Returns:
        pd.DataFrame
    """
    rows = []
    for a in albums:
        if a is None:
            continue
        row = {
            'album_id': a.get('id', ''),
            'album_name': a.get('name', ''),
            'album_type': a.get('album_type', ''),
            'artists': ', '.join([art['name'] for art in a.get('artists', [])]),
            'release_date': a.get('release_date', ''),
            'total_tracks': a.get('total_tracks', 0),
            'spotify_url': a.get('external_urls', {}).get('spotify', ''),
            'image_url': a.get('images', [{}])[0].get('url', '') if a.get('images') else '',
            'uri': a.get('uri', '')
        }
        rows.append(row)
    
    return pd.DataFrame(rows)

In [ ]:
# === Chuyển kết quả thành DataFrame và lưu ===

# Dùng dữ liệu đã thu thập ở phần trước (pop_tracks)
if pop_tracks:
    df_tracks = tracks_to_dataframe(pop_tracks)
    
    print(f"📊 DataFrame shape: {df_tracks.shape}")
    print(f"   Số dòng: {len(df_tracks)} | Số cột: {len(df_tracks.columns)}")
    print(f"\n📋 Các cột:")
    for col in df_tracks.columns:
        print(f"   - {col}: {df_tracks[col].dtype}")
    
    print(f"\n📊 5 dòng đầu tiên:")
    display(df_tracks[['track_name', 'artists', 'album_name', 'popularity', 'duration_s']].head())

In [ ]:
# === Lưu ra CSV ===
output_dir = "spotify_search_api"
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if pop_tracks:
    # Lưu CSV
    csv_path = os.path.join(output_dir, f"tracks_pop_{timestamp}.csv")
    df_tracks.to_csv(csv_path, index=False, encoding='utf-8-sig')  # utf-8-sig để Excel đọc được tiếng Việt
    print(f"✅ Đã lưu CSV: {csv_path}")
    
    # Lưu JSON
    json_path = os.path.join(output_dir, f"tracks_pop_{timestamp}.json")
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(pop_tracks, f, ensure_ascii=False, indent=2)
    print(f"✅ Đã lưu JSON: {json_path}")
    
    # Lưu metadata
    meta_path = os.path.join(output_dir, f"metadata_{timestamp}.json")
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)
    print(f"✅ Đã lưu Metadata: {meta_path}")

---
## 9. Ví dụ thực tế: Thu thập dữ liệu lớn

### Kịch bản: Thu thập tracks theo nhiều thể loại

Vì `offset` bị giới hạn ở 1000, chiến lược để thu thập nhiều dữ liệu hơn là:
1. **Chia nhỏ query**: Tìm theo nhiều từ khóa/thể loại khác nhau
2. **Lọc theo năm**: `year:2020`, `year:2021`, `year:2022`, ...
3. **Loại bỏ trùng lặp**: Dùng `track_id` làm key

In [ ]:
def collect_large_dataset(queries, search_type="track", market="VN",
                           batch_size=50, max_per_query=200, delay=0.3):
    """
    Thu thập dataset lớn bằng cách kết hợp nhiều truy vấn.
    Tự động loại bỏ trùng lặp dựa trên ID.
    
    Args:
        queries (list): Danh sách từ khóa tìm kiếm
        search_type (str): Loại kết quả
        market (str): Mã quốc gia
        batch_size (int): Số items mỗi batch
        max_per_query (int): Số items tối đa cho mỗi query
        delay (float): Thời gian chờ giữa các batch
    
    Returns:
        list: Danh sách items không trùng lặp
    """
    all_items = {}  # Dùng dict để loại trùng theo ID
    
    for i, query in enumerate(queries):
        print(f"\n{'🔴🟡🟢' * 5}")
        print(f"📝 Query {i+1}/{len(queries)}: '{query}'")
        
        items, meta = search_batch_robust(
            query=query,
            search_type=search_type,
            market=market,
            batch_size=batch_size,
            max_items=max_per_query,
            delay=delay
        )
        
        # Thêm vào dict (tự loại trùng)
        before = len(all_items)
        for item in items:
            item_id = item.get('id')
            if item_id and item_id not in all_items:
                all_items[item_id] = item
        
        new_items = len(all_items) - before
        duplicates = len(items) - new_items
        print(f"   📊 Mới: {new_items} | Trùng: {duplicates} | Tổng unique: {len(all_items)}")
        
        # Delay giữa các query
        time.sleep(1)
    
    result = list(all_items.values())
    print(f"\n{'='*70}")
    print(f"🎉 HOÀN TẤT THU THẬP!")
    print(f"   Tổng queries: {len(queries)}")
    print(f"   Tổng items unique: {len(result)}")
    
    return result

In [ ]:
# === Ví dụ: Thu thập V-Pop tracks theo năm ===

# Tạo danh sách queries chia theo năm
vpop_queries = [
    "V-Pop year:2023",
    "V-Pop year:2024",
    "nhạc Việt year:2024",
    "Vietnamese pop year:2024",
]

vpop_dataset = collect_large_dataset(
    queries=vpop_queries,
    search_type="track",
    market="VN",
    batch_size=50,
    max_per_query=100,  # Giảm để demo nhanh, tăng lên 500-1000 khi cần
    delay=0.3
)

In [ ]:
# === Chuyển thành DataFrame và phân tích ===
if vpop_dataset:
    df_vpop = tracks_to_dataframe(vpop_dataset)
    
    print(f"📊 THỐNG KÊ DATASET V-POP:")
    print(f"   Tổng tracks: {len(df_vpop)}")
    print(f"   Popularity trung bình: {df_vpop['popularity'].mean():.1f}")
    print(f"   Popularity cao nhất: {df_vpop['popularity'].max()}")
    print(f"   Duration trung bình: {df_vpop['duration_s'].mean():.0f}s")
    
    # Top 10 bài phổ biến nhất
    print(f"\n🔥 TOP 10 BÀI PHỔ BIẾN NHẤT:")
    top10 = df_vpop.nlargest(10, 'popularity')[['track_name', 'artists', 'popularity', 'release_date']]
    display(top10)
    
    # Lưu ra CSV
    csv_path = os.path.join(output_dir, f"vpop_dataset_{timestamp}.csv")
    df_vpop.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ Đã lưu: {csv_path}")

---
## 📌 Tóm tắt & Best Practices

### Các bước truy xuất dữ liệu theo batch:

```
1. Xác thực → Lấy Access Token
2. Gọi Search API với offset=0, limit=50
3. Đọc trường 'total' để biết tổng số kết quả
4. Thu thập items từ 'items' array
5. Kiểm tra trường 'next':
   - Nếu có URL → gọi tiếp URL đó
   - Nếu null → hết dữ liệu, dừng lại
6. Lặp lại bước 4-5 cho đến khi đủ dữ liệu
```

### Best Practices:

| Practice | Mô tả |
|----------|-------|
| **Dùng `next` URL** | Đáng tin cậy hơn tự tính offset |
| **Batch size = 50** | Giá trị tối đa, giảm số request |
| **Delay 200-300ms** | Tránh rate limiting |
| **Xử lý HTTP 429** | Chờ theo `Retry-After` header |
| **Refresh token** | Token hết hạn sau 1 giờ |
| **Loại trùng bằng ID** | Dùng dict/set với Spotify ID |
| **Chia nhỏ query** | Vượt qua giới hạn offset=1000 |
| **Lưu metadata** | Theo dõi quá trình thu thập |
| **utf-8-sig cho CSV** | Để Excel đọc được tiếng Việt |

### Giới hạn cần nhớ:

| Giới hạn | Giá trị |
|----------|--------|
| `limit` tối đa | 50 |
| `offset` tối đa | 1000 |
| Max kết quả / query | ~1050 (1000 + 50) |
| Token hết hạn | 3600 giây (1 giờ) |
| Rate limit | Không công bố chính xác, ~30 req/s |